# Q-Scanner: Multi-Factor Quantitative Trading Signal Research

A quantitative research project that develops and evaluates technical-factor trading signals across multiple equity assets.

## Research Objective

The objective is to investigate whether a multi-factor technical scoring framework can generate useful trading signals and improve decision-making relative to a simple buy-and-hold benchmark.

## Factors Used

- Moving-average trend
- Relative Strength Index (RSI)
- Bollinger Bands
- Price momentum
- Multi-factor signal scoring

## Research Process

1. Market data acquisition
2. Technical indicator construction
3. Signal generation
4. Iterative strategy development
5. Historical backtesting
6. Transaction-cost modelling
7. Benchmark comparison
8. Risk and performance evaluation

> **Important:** The initial backtest did not outperform buy-and-hold across the tested assets. This project is therefore presented as quantitative strategy research and evaluation, rather than as a claim of a profitable trading system.

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import vectorbt as vbt
import ta

print("Q-SCANNER READY")

Q-SCANNER READY


In [2]:
# ==========================================
# Q-SCANNER — MARKET DATA
# ==========================================

ticker = "AAPL"

data = yf.download(
    ticker,
    period="6mo",
    interval="1d",
    auto_adjust=True,
    progress=False
)

print("Ticker:", ticker)
print("Rows:", len(data))

display(data.tail())

Ticker: AAPL
Rows: 128


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2026-08-31,316.850006,321.239990,312.799988,319.600006,41242700
2026-09-01,325.130005,327.299988,314.730011,316.980011,53167400
2026-09-02,324.959991,328.399994,323.529999,326.869995,33776400
2026-09-03,328.209991,330.809998,324.109985,324.869995,37225800
2026-09-04,319.970001,328.929993,317.859985,328.309998,39551800


In [3]:
# ==========================================
# Q-SCANNER — TECHNICAL INDICATORS
# ==========================================

from ta.trend import SMAIndicator
from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands

# Make sure we are working with a Series
close = data["Close"].squeeze()

# Moving averages
data["SMA_20"] = SMAIndicator(
    close=close,
    window=20
).sma_indicator()

data["SMA_50"] = SMAIndicator(
    close=close,
    window=50
).sma_indicator()

# RSI
data["RSI"] = RSIIndicator(
    close=close,
    window=14
).rsi()

# Bollinger Bands
bb = BollingerBands(
    close=close,
    window=20,
    window_dev=2
)

data["BB_Upper"] = bb.bollinger_hband()
data["BB_Middle"] = bb.bollinger_mavg()
data["BB_Lower"] = bb.bollinger_lband()

print("Technical indicators calculated successfully.")

display(
    data[
        [
            "Close",
            "SMA_20",
            "SMA_50",
            "RSI",
            "BB_Upper",
            "BB_Middle",
            "BB_Lower"
        ]
    ].tail(10)
)

Technical indicators calculated successfully.


Price,Close,SMA_20,SMA_50,RSI,BB_Upper,BB_Middle,BB_Lower
Ticker,AAPL,,,,,,
Date,,,,,,,
2026-08-24,310.339996,312.886340,310.293065,48.349004,334.309812,312.886340,291.462869
2026-08-25,309.899994,311.391994,310.673482,47.934670,328.915925,311.391994,293.868062
2026-08-26,313.450012,310.169565,311.019191,51.542800,322.875391,310.169565,297.463739
2026-08-27,314.579987,309.241431,311.331147,52.667194,316.746757,309.241431,301.736104
2026-08-28,319.700012,309.794240,311.811248,57.481427,318.564217,309.794240,301.024263
2026-08-31,316.850006,310.478812,312.193184,54.178133,319.207402,310.478812,301.750221
2026-09-01,325.130005,311.279642,312.760703,61.161315,322.058398,311.279642,300.500885
2026-09-02,324.959991,311.991040,313.378975,60.955899,324.300697,311.991040,299.681383


## Signal Development

The Q-Scanner is being developed iteratively. Multiple versions of the signal engine are retained to document the progression of the research methodology.

The development process explores different approaches to:

- Trend identification using moving averages
- Momentum measurement
- RSI-based market conditions
- Bollinger Band positioning
- Multi-factor scoring
- Signal confidence
- Risk-management considerations

Earlier versions are retained as part of the research history rather than being removed, allowing later versions to be compared against previous approaches.

In [8]:
# ==========================================
# Q-SCANNER — SIGNAL ENGINE
# ==========================================

latest = data.iloc[-1]

# Convert single-value Series to normal numbers
price = float(np.asarray(latest["Close"]).squeeze())
sma20 = float(np.asarray(latest["SMA_20"]).squeeze())
sma50 = float(np.asarray(latest["SMA_50"]).squeeze())
rsi = float(np.asarray(latest["RSI"]).squeeze())

# Signal rules
if price > sma20 and sma20 > sma50 and 50 < rsi < 70:
    signal = "BUY"

elif price < sma20 and sma20 < sma50 and 30 < rsi < 50:
    signal = "SELL"

else:
    signal = "NEUTRAL"

print("================================")
print("        Q-SCANNER SIGNAL")
print("================================")
print(f"Price:       {price:.2f}")
print(f"SMA 20:      {sma20:.2f}")
print(f"SMA 50:      {sma50:.2f}")
print(f"RSI:         {rsi:.2f}")
print("--------------------------------")
print(f"SIGNAL:      {signal}")
print("================================")

        Q-SCANNER SIGNAL
Price:       319.70
SMA 20:      309.79
SMA 50:      311.81
RSI:         57.48
--------------------------------
SIGNAL:      NEUTRAL


In [9]:
# ==========================================
# Q-SCANNER — SCORING ENGINE
# ==========================================

score = 0
max_score = 6

# --------------------------
# 1. TREND
# --------------------------

if price > sma20:
    score += 1
    trend_price = "Bullish"
else:
    trend_price = "Bearish"

if sma20 > sma50:
    score += 1
    trend_ma = "Bullish"
else:
    trend_ma = "Bearish"


# --------------------------
# 2. MOMENTUM — RSI
# --------------------------

if 50 <= rsi < 70:
    score += 2
    momentum = "Strong"
elif rsi >= 70:
    momentum = "Overbought"
elif 30 <= rsi < 50:
    momentum = "Weak"
else:
    momentum = "Oversold"


# --------------------------
# 3. BOLLINGER BANDS
# --------------------------

bb_upper = float(np.asarray(latest["BB_Upper"]).squeeze())
bb_lower = float(np.asarray(latest["BB_Lower"]).squeeze())

if price < bb_lower:
    score += 1
    volatility_signal = "Potentially Oversold"

elif price > bb_upper:
    volatility_signal = "Potentially Overbought"

else:
    score += 1
    volatility_signal = "Normal"


# --------------------------
# 4. FINAL CLASSIFICATION
# --------------------------

if score >= 5:
    final_signal = "STRONG BUY"

elif score >= 4:
    final_signal = "BUY"

elif score <= 1:
    final_signal = "SELL"

else:
    final_signal = "NEUTRAL"


# --------------------------
# DISPLAY RESULT
# --------------------------

print("========================================")
print("          Q-SCANNER ANALYSIS")
print("========================================")
print(f"Price:              {price:.2f}")
print(f"SMA 20:             {sma20:.2f}")
print(f"SMA 50:             {sma50:.2f}")
print(f"RSI:                {rsi:.2f}")
print("----------------------------------------")
print(f"Price vs SMA20:     {trend_price}")
print(f"SMA20 vs SMA50:     {trend_ma}")
print(f"Momentum:           {momentum}")
print(f"Bollinger Signal:   {volatility_signal}")
print("----------------------------------------")
print(f"SCORE:              {score}/{max_score}")
print(f"FINAL SIGNAL:       {final_signal}")
print("========================================")

          Q-SCANNER ANALYSIS
Price:              319.70
SMA 20:             309.79
SMA 50:             311.81
RSI:                57.48
----------------------------------------
Price vs SMA20:     Bullish
SMA20 vs SMA50:     Bearish
Momentum:           Strong
Bollinger Signal:   Potentially Overbought
----------------------------------------
SCORE:              3/6
FINAL SIGNAL:       NEUTRAL


In [10]:
# ============================================================
# Q-SCANNER — MULTI-ASSET SCANNER
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import ta

# Assets to scan
tickers = [
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "META",
    "GOOGL",
    "TSLA"
]

results = []

for ticker in tickers:

    try:
        # Download market data
        data = yf.download(
            ticker,
            period="6mo",
            interval="1d",
            auto_adjust=True,
            progress=False
        )

        # Skip if no data
        if data.empty:
            print(f"{ticker}: No data")
            continue

        # Handle possible MultiIndex columns
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)

        # Calculate indicators
        data["SMA_20"] = ta.trend.SMAIndicator(
            close=data["Close"],
            window=20
        ).sma_indicator()

        data["SMA_50"] = ta.trend.SMAIndicator(
            close=data["Close"],
            window=50
        ).sma_indicator()

        data["RSI"] = ta.momentum.RSIIndicator(
            close=data["Close"],
            window=14
        ).rsi()

        bb = ta.volatility.BollingerBands(
            close=data["Close"],
            window=20,
            window_dev=2
        )

        data["BB_Upper"] = bb.bollinger_hband()
        data["BB_Middle"] = bb.bollinger_mavg()
        data["BB_Lower"] = bb.bollinger_lband()

        # Remove incomplete rows
        data = data.dropna()

        if data.empty:
            continue

        latest = data.iloc[-1]

        # Convert values to normal numbers
        price = float(np.asarray(latest["Close"]).squeeze())
        sma20 = float(np.asarray(latest["SMA_20"]).squeeze())
        sma50 = float(np.asarray(latest["SMA_50"]).squeeze())
        rsi = float(np.asarray(latest["RSI"]).squeeze())
        bb_upper = float(np.asarray(latest["BB_Upper"]).squeeze())
        bb_lower = float(np.asarray(latest["BB_Lower"]).squeeze())

        # ====================================================
        # SIGNAL SCORING
        # ====================================================

        score = 0
        max_score = 6

        # 1. Price vs SMA20
        if price > sma20:
            trend_price = "Bullish"
            score += 1
        else:
            trend_price = "Bearish"

        # 2. SMA20 vs SMA50
        if sma20 > sma50:
            trend_ma = "Bullish"
            score += 1
        else:
            trend_ma = "Bearish"

        # 3. RSI / Momentum
        if 50 < rsi < 70:
            momentum = "Strong"
            score += 1
        elif 30 < rsi <= 50:
            momentum = "Weak"
        elif rsi >= 70:
            momentum = "Overbought"
        else:
            momentum = "Oversold"
            score += 1

        # 4. Bollinger Bands
        if price < bb_lower:
            volatility_signal = "Potentially Oversold"
            score += 2
        elif price > bb_upper:
            volatility_signal = "Potentially Overbought"
        else:
            volatility_signal = "Normal"

        # 5. Final signal
        if score >= 5:
            final_signal = "STRONG BUY"
        elif score == 4:
            final_signal = "BUY"
        elif score <= 1:
            final_signal = "SELL"
        else:
            final_signal = "NEUTRAL"

        # Store results
        results.append({
            "Ticker": ticker,
            "Price": round(price, 2),
            "SMA_20": round(sma20, 2),
            "SMA_50": round(sma50, 2),
            "RSI": round(rsi, 2),
            "Price Trend": trend_price,
            "MA Trend": trend_ma,
            "Momentum": momentum,
            "Bollinger": volatility_signal,
            "Score": score,
            "Signal": final_signal
        })

    except Exception as e:
        print(f"{ticker}: ERROR -> {e}")


# ============================================================
# CREATE SCANNER TABLE
# ============================================================

scanner_results = pd.DataFrame(results)

if not scanner_results.empty:

    # Rank strongest signals first
    scanner_results = scanner_results.sort_values(
        by=["Score", "RSI"],
        ascending=[False, False]
    ).reset_index(drop=True)

    print("=" * 100)
    print("                 Q-SCANNER — MULTI-ASSET ANALYSIS")
    print("=" * 100)

    display(scanner_results)

else:
    print("No assets were successfully scanned.")

                 Q-SCANNER — MULTI-ASSET ANALYSIS


,Ticker,Price,SMA_20,SMA_50,RSI,Price Trend,MA Trend,Momentum,Bollinger,Score,Signal
0,MSFT,513.53,492.75,429.97,73.54,Bullish,Bullish,Overbought,Potentially Overbought,2,NEUTRAL
1,AAPL,319.70,309.79,311.81,57.48,Bullish,Bearish,Strong,Potentially Overbought,2,NEUTRAL
2,AMZN,266.43,266.88,251.66,55.85,Bearish,Bullish,Strong,Normal,2,NEUTRAL
3,NVDA,217.55,218.05,208.42,52.34,Bearish,Bullish,Strong,Normal,2,NEUTRAL
4,TSLA,348.75,338.82,360.47,50.89,Bullish,Bearish,Strong,Normal,2,NEUTRAL
5,META,578.02,575.77,592.14,49.56,Bullish,Bearish,Weak,Normal,1,SELL
6,GOOGL,346.59,350.27,349.78,49.26,Bearish,Bullish,Weak,Normal,1,SELL


## Iterative Strategy Development

The signal engine is intentionally developed through successive versions rather than treating the first set of rules as final.

Each iteration modifies the factor scoring methodology in an attempt to improve signal quality, interpretability, and robustness.

The progression is:

**V1 → V2 → V3 → V4**

- **V1:** Initial multi-factor scoring and historical backtesting
- **V2:** Introduces positive/negative factor scoring and confidence estimation
- **V3:** Refines RSI, momentum, and Bollinger Band scoring
- **V4:** Extends the framework with enhanced signal logic and configurable risk-management parameters

The versions are retained to preserve the research history and allow future comparison between alternative specifications.

In [11]:
# ============================================================
# Q-SCANNER — IMPROVED SIGNAL ENGINE v2
# ============================================================

results_v2 = []

for ticker in tickers:

    try:
        # Download market data
        data = yf.download(
            ticker,
            period="6mo",
            interval="1d",
            auto_adjust=True,
            progress=False
        )

        if data.empty:
            continue

        # Handle MultiIndex columns
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)

        # ----------------------------------------------------
        # TECHNICAL INDICATORS
        # ----------------------------------------------------

        data["SMA_20"] = ta.trend.SMAIndicator(
            close=data["Close"],
            window=20
        ).sma_indicator()

        data["SMA_50"] = ta.trend.SMAIndicator(
            close=data["Close"],
            window=50
        ).sma_indicator()

        data["RSI"] = ta.momentum.RSIIndicator(
            close=data["Close"],
            window=14
        ).rsi()

        bb = ta.volatility.BollingerBands(
            close=data["Close"],
            window=20,
            window_dev=2
        )

        data["BB_Upper"] = bb.bollinger_hband()
        data["BB_Middle"] = bb.bollinger_mavg()
        data["BB_Lower"] = bb.bollinger_lband()

        data = data.dropna()

        if data.empty:
            continue

        latest = data.iloc[-1]

        # Convert values to numbers
        price = float(np.asarray(latest["Close"]).squeeze())
        sma20 = float(np.asarray(latest["SMA_20"]).squeeze())
        sma50 = float(np.asarray(latest["SMA_50"]).squeeze())
        rsi = float(np.asarray(latest["RSI"]).squeeze())
        bb_upper = float(np.asarray(latest["BB_Upper"]).squeeze())
        bb_lower = float(np.asarray(latest["BB_Lower"]).squeeze())

        # ====================================================
        # IMPROVED SCORING
        # Maximum = +6
        # Minimum = -6
        # ====================================================

        score = 0

        # ----------------------------------------------------
        # 1. PRICE VS SMA20
        # ----------------------------------------------------

        if price > sma20:
            price_trend = "Bullish"
            score += 1
        else:
            price_trend = "Bearish"
            score -= 1

        # ----------------------------------------------------
        # 2. SMA20 VS SMA50
        # ----------------------------------------------------

        if sma20 > sma50:
            moving_average_trend = "Bullish"
            score += 1
        else:
            moving_average_trend = "Bearish"
            score -= 1

        # ----------------------------------------------------
        # 3. RSI / MOMENTUM
        # ----------------------------------------------------

        if 50 <= rsi < 70:
            momentum = "Bullish"
            score += 1

        elif 70 <= rsi:
            momentum = "Overbought"
            score -= 1

        elif 30 < rsi < 50:
            momentum = "Bearish"
            score -= 1

        else:
            momentum = "Oversold"
            score += 1

        # ----------------------------------------------------
        # 4. BOLLINGER BANDS
        # ----------------------------------------------------

        if price < bb_lower:
            bollinger_signal = "Oversold"
            score += 2

        elif price > bb_upper:
            bollinger_signal = "Overbought"
            score -= 2

        else:
            bollinger_signal = "Normal"

        # ----------------------------------------------------
        # 5. FINAL SIGNAL
        # ----------------------------------------------------

        if score >= 4:
            final_signal = "STRONG BUY"

        elif score >= 2:
            final_signal = "BUY"

        elif score <= -4:
            final_signal = "STRONG SELL"

        elif score <= -2:
            final_signal = "SELL"

        else:
            final_signal = "NEUTRAL"

        # ----------------------------------------------------
        # CONFIDENCE
        # ----------------------------------------------------

        confidence = round((abs(score) / 6) * 100, 1)

        # ----------------------------------------------------
        # SAVE RESULT
        # ----------------------------------------------------

        results_v2.append({
            "Ticker": ticker,
            "Price": round(price, 2),
            "SMA_20": round(sma20, 2),
            "SMA_50": round(sma50, 2),
            "RSI": round(rsi, 2),
            "Price Trend": price_trend,
            "MA Trend": moving_average_trend,
            "Momentum": momentum,
            "Bollinger": bollinger_signal,
            "Score": score,
            "Confidence %": confidence,
            "Signal": final_signal
        })

    except Exception as e:
        print(f"{ticker}: ERROR -> {e}")


# ============================================================
# CREATE IMPROVED SCANNER TABLE
# ============================================================

scanner_v2 = pd.DataFrame(results_v2)

if not scanner_v2.empty:

    # Strongest signals first
    scanner_v2["Signal Rank"] = scanner_v2["Score"].abs()

    scanner_v2 = scanner_v2.sort_values(
        by=["Score", "RSI"],
        ascending=[False, False]
    ).reset_index(drop=True)

    # Remove helper column
    scanner_v2 = scanner_v2.drop(columns=["Signal Rank"])

    print("=" * 110)
    print("              Q-SCANNER — IMPROVED MULTI-ASSET ANALYSIS")
    print("=" * 110)

    display(scanner_v2)

else:
    print("No assets were successfully scanned.")

              Q-SCANNER — IMPROVED MULTI-ASSET ANALYSIS


,Ticker,Price,SMA_20,SMA_50,RSI,Price Trend,MA Trend,Momentum,Bollinger,Score,Confidence %,Signal
0,AMZN,266.43,266.88,251.66,55.85,Bearish,Bullish,Bullish,Normal,1,16.7,NEUTRAL
1,NVDA,217.55,218.05,208.42,52.34,Bearish,Bullish,Bullish,Normal,1,16.7,NEUTRAL
2,TSLA,348.75,338.82,360.47,50.89,Bullish,Bearish,Bullish,Normal,1,16.7,NEUTRAL
3,MSFT,513.53,492.75,429.97,73.54,Bullish,Bullish,Overbought,Overbought,-1,16.7,NEUTRAL
4,AAPL,319.70,309.79,311.81,57.48,Bullish,Bearish,Bullish,Overbought,-1,16.7,NEUTRAL
5,META,578.02,575.77,592.14,49.56,Bullish,Bearish,Bearish,Normal,-1,16.7,NEUTRAL
6,GOOGL,346.59,350.27,349.78,49.26,Bearish,Bullish,Bearish,Normal,-1,16.7,NEUTRAL


In [13]:
# ============================================================
# Q-SCANNER — SIGNAL ENGINE V3
# ============================================================

def generate_signal(row):

    score = 0
    reasons = []

    # --------------------------------------------------------
    # 1. PRICE VS SMA 20
    # --------------------------------------------------------
    if row["Price"] > row["SMA_20"]:
        score += 1
        reasons.append("Price above SMA20")
    else:
        score -= 1
        reasons.append("Price below SMA20")

    # --------------------------------------------------------
    # 2. SMA 20 VS SMA 50
    # --------------------------------------------------------
    if row["SMA_20"] > row["SMA_50"]:
        score += 1
        reasons.append("SMA20 above SMA50")
    else:
        score -= 1
        reasons.append("SMA20 below SMA50")

    # --------------------------------------------------------
    # 3. RSI
    # --------------------------------------------------------
    rsi = float(row["RSI"])

    if 50 <= rsi < 65:
        score += 1
        reasons.append("Healthy bullish RSI")

    elif 65 <= rsi < 70:
        score += 0.5
        reasons.append("Strong RSI")

    elif 30 < rsi < 50:
        score -= 1
        reasons.append("Bearish RSI")

    elif rsi <= 30:
        score += 0.5
        reasons.append("Potential RSI reversal")

    elif rsi >= 70:
        score -= 1
        reasons.append("RSI overbought")

    # --------------------------------------------------------
    # 4. MOMENTUM
    # --------------------------------------------------------
    momentum = str(row["Momentum"])

    if momentum == "Bullish":
        score += 1
        reasons.append("Bullish momentum")

    elif momentum == "Bearish":
        score -= 1
        reasons.append("Bearish momentum")

    # --------------------------------------------------------
    # 5. BOLLINGER
    # --------------------------------------------------------
    bollinger = str(row["Bollinger"])

    if bollinger == "Potentially Overbought":
        score -= 0.5
        reasons.append("Potentially overbought")

    elif bollinger == "Potentially Oversold":
        score += 0.5
        reasons.append("Potentially oversold")

    # --------------------------------------------------------
    # FINAL SIGNAL
    # --------------------------------------------------------
    if score >= 3:
        signal = "STRONG BUY"

    elif score >= 2:
        signal = "BUY"

    elif score <= -3:
        signal = "STRONG SELL"

    elif score <= -2:
        signal = "SELL"

    else:
        signal = "NEUTRAL"

    # --------------------------------------------------------
    # CONFIDENCE
    # --------------------------------------------------------
    max_score = 5.5
    confidence = min(abs(score) / max_score * 100, 100)

    return pd.Series({
        "New Score": round(score, 2),
        "Confidence %": round(confidence, 1),
        "New Signal": signal,
        "Reasons": "; ".join(reasons)
    })


# ============================================================
# CREATE V3 TABLE
# ============================================================

# Remove old scoring columns if they already exist
base = scanner_v2.copy()

for column in ["Score", "Confidence %", "Signal", "Reasons", "Signal Rank"]:
    if column in base.columns:
        base = base.drop(columns=[column])


# Generate new signals
signal_output = base.apply(generate_signal, axis=1)

# Combine everything
scanner_v3 = pd.concat(
    [base, signal_output],
    axis=1
)


# ============================================================
# RANK STRONGEST SIGNALS
# ============================================================

scanner_v3["Signal Rank"] = scanner_v3["New Score"].abs()

scanner_v3 = scanner_v3.sort_values(
    by=["Signal Rank", "Confidence %"],
    ascending=[False, False]
).reset_index(drop=True)


# Remove helper column
scanner_v3 = scanner_v3.drop(columns=["Signal Rank"])


# ============================================================
# DISPLAY
# ============================================================

print("=" * 120)
print("Q-SCANNER — SIGNAL ENGINE V3")
print("=" * 120)

display(scanner_v3)

Q-SCANNER — SIGNAL ENGINE V3


,Ticker,Price,SMA_20,SMA_50,RSI,Price Trend,MA Trend,Momentum,Bollinger,New Score,Confidence %,New Signal,Reasons
0,AMZN,266.43,266.88,251.66,55.85,Bearish,Bullish,Bullish,Normal,2,36.4,BUY,Price below SMA20; SMA20 above SMA50; Healthy ...
1,NVDA,217.55,218.05,208.42,52.34,Bearish,Bullish,Bullish,Normal,2,36.4,BUY,Price below SMA20; SMA20 above SMA50; Healthy ...
2,TSLA,348.75,338.82,360.47,50.89,Bullish,Bearish,Bullish,Normal,2,36.4,BUY,Price above SMA20; SMA20 below SMA50; Healthy ...
3,AAPL,319.70,309.79,311.81,57.48,Bullish,Bearish,Bullish,Overbought,2,36.4,BUY,Price above SMA20; SMA20 below SMA50; Healthy ...
4,META,578.02,575.77,592.14,49.56,Bullish,Bearish,Bearish,Normal,-2,36.4,SELL,Price above SMA20; SMA20 below SMA50; Bearish ...
5,GOOGL,346.59,350.27,349.78,49.26,Bearish,Bullish,Bearish,Normal,-2,36.4,SELL,Price below SMA20; SMA20 above SMA50; Bearish ...
6,MSFT,513.53,492.75,429.97,73.54,Bullish,Bullish,Overbought,Overbought,1,18.2,NEUTRAL,Price above SMA20; SMA20 above SMA50; RSI over...


## Historical Backtest — V1

The first backtesting framework evaluates the multi-factor signal strategy over approximately five years of historical data across seven US equities.

### Evaluation

The strategy is evaluated using:

- Strategy cumulative return
- Buy-and-hold cumulative return
- Win rate on active trading days
- Maximum drawdown
- Number of signal days
- Transaction-cost modelling

A transaction cost of 0.10% is applied when the simulated position changes.

The strategy is compared against a buy-and-hold benchmark to determine whether the systematic signal framework provides an improvement over passive exposure.

### Research Interpretation

The initial results show that the strategy does not consistently outperform buy-and-hold across the tested assets. This is treated as a research finding rather than a failure of the project.

The result provides evidence that the current factor combination and signal rules require further investigation and motivates subsequent iterations of the signal engine.

In [14]:
# ============================================================
# Q-SCANNER — HISTORICAL BACKTEST ENGINE V1
# ============================================================

import yfinance as yf
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# ASSETS TO BACKTEST
# ------------------------------------------------------------

tickers = [
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "META",
    "GOOGL",
    "TSLA"
]


# ------------------------------------------------------------
# BACKTEST SETTINGS
# ------------------------------------------------------------

period = "5y"

transaction_cost = 0.001   # 0.10% per trade


# ------------------------------------------------------------
# FUNCTION: CALCULATE INDICATORS
# ------------------------------------------------------------

def calculate_indicators(df):

    df = df.copy()

    # Simple Moving Averages
    df["SMA_20"] = df["Close"].rolling(20).mean()
    df["SMA_50"] = df["Close"].rolling(50).mean()

    # RSI
    delta = df["Close"].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss

    df["RSI"] = 100 - (100 / (1 + rs))

    # Bollinger Bands
    df["BB_Middle"] = df["Close"].rolling(20).mean()
    df["BB_STD"] = df["Close"].rolling(20).std()

    df["BB_Upper"] = (
        df["BB_Middle"] + 2 * df["BB_STD"]
    )

    df["BB_Lower"] = (
        df["BB_Middle"] - 2 * df["BB_STD"]
    )

    # Momentum
    df["Momentum_Value"] = df["Close"].pct_change(10)

    return df


# ------------------------------------------------------------
# FUNCTION: GENERATE SCORE
# ------------------------------------------------------------

def calculate_score(row):

    score = 0

    price = row["Close"]
    sma20 = row["SMA_20"]
    sma50 = row["SMA_50"]
    rsi = row["RSI"]

    # Price vs SMA20
    if price > sma20:
        score += 1
    else:
        score -= 1

    # SMA20 vs SMA50
    if sma20 > sma50:
        score += 1
    else:
        score -= 1

    # RSI
    if 50 <= rsi < 65:
        score += 1

    elif 65 <= rsi < 70:
        score += 0.5

    elif 30 < rsi < 50:
        score -= 1

    elif rsi <= 30:
        score += 0.5

    elif rsi >= 70:
        score -= 1

    # Momentum
    if row["Momentum_Value"] > 0:
        score += 1
    else:
        score -= 1

    # Bollinger condition
    if price > row["BB_Upper"]:
        score -= 0.5

    elif price < row["BB_Lower"]:
        score += 0.5

    return score


# ------------------------------------------------------------
# BACKTEST ONE ASSET
# ------------------------------------------------------------

def backtest_ticker(ticker):

    print(f"Backtesting {ticker}...")

    df = yf.download(
        ticker,
        period=period,
        interval="1d",
        auto_adjust=True,
        progress=False
    )

    if df.empty:
        return None

    # Handle yfinance multi-level columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = calculate_indicators(df)

    # Remove incomplete rows
    df = df.dropna().copy()

    # Calculate today's score
    df["Score"] = df.apply(calculate_score, axis=1)

    # --------------------------------------------------------
    # SIGNAL
    # --------------------------------------------------------

    df["Signal"] = "NEUTRAL"

    df.loc[df["Score"] >= 2, "Signal"] = "BUY"
    df.loc[df["Score"] <= -2, "Signal"] = "SELL"

    # --------------------------------------------------------
    # POSITION
    # --------------------------------------------------------

    df["Position"] = 0

    df.loc[df["Score"] >= 2, "Position"] = 1
    df.loc[df["Score"] <= -2, "Position"] = -1

    # --------------------------------------------------------
    # NEXT-DAY RETURN
    # --------------------------------------------------------

    df["Market_Return"] = df["Close"].pct_change().shift(-1)

    # Signal generated today acts on the NEXT trading day
    df["Strategy_Return"] = (
        df["Position"] * df["Market_Return"]
    )

    # --------------------------------------------------------
    # TRANSACTION COST
    # --------------------------------------------------------

    df["Position_Change"] = (
        df["Position"].diff().abs()
    )

    df["Strategy_Return"] = (
        df["Strategy_Return"]
        - df["Position_Change"] * transaction_cost
    )

    # Remove final row
    df = df.dropna(subset=["Strategy_Return"])

    # --------------------------------------------------------
    # PERFORMANCE
    # --------------------------------------------------------

    cumulative_return = (
        (1 + df["Strategy_Return"]).prod() - 1
    )

    buy_hold_return = (
        (1 + df["Market_Return"]).prod() - 1
    )

    # Winning trading days
    active_trades = df[
        df["Position"] != 0
    ]

    if len(active_trades) > 0:

        win_rate = (
            active_trades["Strategy_Return"] > 0
        ).mean() * 100

    else:
        win_rate = 0

    # Maximum drawdown
    equity_curve = (
        1 + df["Strategy_Return"]
    ).cumprod()

    running_max = equity_curve.cummax()

    drawdown = (
        equity_curve / running_max
    ) - 1

    max_drawdown = drawdown.min()

    # Number of signal days
    signal_days = (
        df["Position"] != 0
    ).sum()

    # --------------------------------------------------------
    # RESULT
    # --------------------------------------------------------

    result = {
        "Ticker": ticker,
        "Strategy Return %": round(
            cumulative_return * 100, 2
        ),
        "Buy & Hold %": round(
            buy_hold_return * 100, 2
        ),
        "Win Rate %": round(
            win_rate, 2
        ),
        "Max Drawdown %": round(
            max_drawdown * 100, 2
        ),
        "Signal Days": int(signal_days)
    }

    return result


# ============================================================
# RUN BACKTEST
# ============================================================

backtest_results = []

for ticker in tickers:

    result = backtest_ticker(ticker)

    if result is not None:
        backtest_results.append(result)


# ------------------------------------------------------------
# CREATE RESULTS TABLE
# ------------------------------------------------------------

backtest_df = pd.DataFrame(backtest_results)

if not backtest_df.empty:

    backtest_df = backtest_df.sort_values(
        by="Strategy Return %",
        ascending=False
    ).reset_index(drop=True)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print()
print("=" * 100)
print("Q-SCANNER — HISTORICAL BACKTEST RESULTS")
print("=" * 100)

display(backtest_df)

Backtesting AAPL...
Backtesting MSFT...
Backtesting NVDA...
Backtesting AMZN...
Backtesting META...
Backtesting GOOGL...
Backtesting TSLA...

Q-SCANNER — HISTORICAL BACKTEST RESULTS


,Ticker,Strategy Return %,Buy & Hold %,Win Rate %,Max Drawdown %,Signal Days
0,AAPL,28.75,117.02,50.93,-35.69,911
1,META,2.22,73.87,50.83,-51.63,905
2,GOOGL,-4.78,134.80,51.00,-53.76,904
3,NVDA,-21.80,611.95,47.84,-71.72,928
4,AMZN,-48.45,49.00,49.08,-77.26,925
5,MSFT,-55.26,59.27,49.11,-63.67,902
6,TSLA,-78.42,2.22,48.27,-85.74,955


In [1]:
# ============================================================
# Q-SCANNER V4 — STRATEGY + RISK MANAGEMENT BACKTEST ## V4
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
from ta.trend import SMAIndicator
from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

TICKERS = ["AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "TSLA"]

START_DATE = "2022-01-01"
END_DATE = None

INITIAL_CAPITAL = 10_000

STOP_LOSS = 0.03       # 3%
TAKE_PROFIT = 0.06     # 6%

RSI_BUY_MIN = 50
RSI_BUY_MAX = 70

RSI_SELL_MIN = 30
RSI_SELL_MAX = 50


# ------------------------------------------------------------
# V4 SIGNAL FUNCTION
# ------------------------------------------------------------

def generate_v4_signal(df):

    df = df.copy()

    # Technical indicators
    df["SMA_20"] = SMAIndicator(
        close=df["Close"],
        window=20
    ).sma_indicator()

    df["SMA_50"] = SMAIndicator(
        close=df["Close"],
        window=50
    ).sma_indicator()

    df["RSI"] = RSIIndicator(
        close=df["Close"],
        window=14
    ).rsi()

    bb = BollingerBands(
        close=df["Close"],
        window=20,
        window_dev=2
    )

    df["BB_Upper"] = bb.bollinger_hband()
    df["BB_Middle"] = bb.bollinger_mavg()
    df["BB_Lower"] = bb.bollinger_lband()

    # Momentum
    df["Momentum"] = df["Close"].pct_change(10) * 100

    # Remove incomplete rows
    df = df.dropna().copy()

    # --------------------------------------------------------
    # SCORING SYSTEM
    # --------------------------------------------------------

    scores = []

    for i in range(len(df)):

        price = float(df["Close"].iloc[i])
        sma20 = float(df["SMA_20"].iloc[i])
        sma50 = float(df["SMA_50"].iloc[i])
        rsi = float(df["RSI"].iloc[i])
        bb_upper = float(df["BB_Upper"].iloc[i])
        bb_lower = float(df["BB_Lower"].iloc[i])
        momentum = float(df["Momentum"].iloc[i])

        # =========================
        # SIGNAL SCORING
        # =========================

        score = 0
        reasons = []

        # Price vs SMA20
        if price > sma20:
            score += 1
            reasons.append("Price above SMA20")
        else:
            score -= 1
            reasons.append("Price below SMA20")

        # SMA20 vs SMA50
        if sma20 > sma50:
            score += 1
            reasons.append("SMA20 above SMA50")
        else:
            score -= 1
            reasons.append("SMA20 below SMA50")

        # RSI
        if 50 <= rsi <= 70:
            score += 1
            reasons.append("RSI bullish")
        elif 30 <= rsi < 50:
            score -= 1
            reasons.append("RSI bearish")
        elif rsi > 70:
            score -= 1
            reasons.append("RSI overbought")
        else:
            score += 1
            reasons.append("RSI oversold")

        # Momentum
        if momentum > 0:
            score += 1
            reasons.append("Positive momentum")
        else:
            score -= 1
            reasons.append("Negative momentum")

        # Bollinger Bands
        if price >= bb_upper:
            score -= 1
            reasons.append("Price near/above upper Bollinger Band")
        elif price <= bb_lower:
            score += 1
            reasons.append("Price near/below lower Bollinger Band")
        else:
            reasons.append("Price inside Bollinger Bands")

        # =========================
        # FINAL SIGNAL
        # =========================

        if score >= 3:
            signal = "BUY"
        elif score <= -3:
            signal = "SELL"
        else:
            signal = "NEUTRAL"

        scores.append({
            "Score": score,
            "Signal": signal,
            "Reasons": "; ".join(reasons)
        })

    # =========================
    # CREATE RESULTS TABLE
    # =========================

    score_df = pd.DataFrame(scores)

    df = df.reset_index(drop=True)

    scanner_v3 = df.copy()

    scanner_v3["Score"] = score_df["Score"]
    scanner_v3["Signal"] = score_df["Signal"]
    scanner_v3["Reasons"] = score_df["Reasons"]

    # =========================
    # DISPLAY RESULTS
    # =========================

    print("=" * 120)
    print("Q-SCANNER — SIGNAL ENGINE V3")
    print("=" * 120)

    display(
        scanner_v3[
            [
                "Score",
                "Signal",
                "Reasons"
            ]
        ].tail(20)
    )

    return scanner_v3

### Development Note

This version extends the multi-factor signal framework by introducing configurable risk-management parameters, including stop-loss and take-profit thresholds.

At this stage, these parameters are defined for further development but are **not yet incorporated into the historical trade simulation**. The backtesting framework therefore does not currently claim to evaluate the effectiveness of the stop-loss or take-profit rules.

This section represents an ongoing stage of strategy development.

In [29]:
# ============================================================
# RUN Q-SCANNER V4
# ============================================================

results = []

for ticker in TICKERS:

    print(f"Scanning {ticker}...")

    try:
        # Download historical data
        data = yf.download(
            ticker,
            start=START_DATE,
            end=END_DATE,
            auto_adjust=True,
            progress=False
        )

        # Make sure data exists
        if data.empty:
            print(f"No data found for {ticker}")
            continue

        # Handle newer yfinance MultiIndex columns
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)

        # Generate signals
        scanned = generate_v4_signal(data)

        # Get latest row
        latest = scanned.iloc[-1].copy()

        # Add ticker
        latest["Ticker"] = ticker

        results.append(latest)

    except Exception as e:
        print(f"Error scanning {ticker}: {e}")


# ============================================================
# CREATE FINAL SCANNER TABLE
# ============================================================

if results:

    scanner_results = pd.DataFrame(results)

    # Rank strongest signals first
    scanner_results["Signal Strength"] = (
        scanner_results["Score"].abs()
    )

    scanner_results = scanner_results.sort_values(
        by=["Signal Strength", "Score"],
        ascending=[False, False]
    ).reset_index(drop=True)

    # ========================================================
    # DISPLAY
    # ========================================================

    print()
    print("=" * 120)
    print("Q-SCANNER V4 — CURRENT MARKET SIGNALS")
    print("=" * 120)

    display(
        scanner_results[
            [
                "Ticker",
                "Close",
                "SMA_20",
                "SMA_50",
                "RSI",
                "Momentum",
                "Score",
                "Signal",
                "Reasons"
            ]
        ]
    )
   
else:

    print("No assets were successfully scanned.")

Scanning AAPL...
Q-SCANNER — SIGNAL ENGINE V3


Price,Score,Signal,Reasons
1099,-2,NEUTRAL,Price below SMA20; SMA20 above SMA50; RSI bear...
1100,-2,NEUTRAL,Price below SMA20; SMA20 above SMA50; RSI bear...
1101,-2,NEUTRAL,Price below SMA20; SMA20 above SMA50; RSI bear...
1102,-2,NEUTRAL,Price below SMA20; SMA20 above SMA50; RSI bear...
1103,-2,NEUTRAL,Price below SMA20; SMA20 above SMA50; RSI bear...
1104,-2,NEUTRAL,Price below SMA20; SMA20 above SMA50; RSI bear...
1105,-2,NEUTRAL,Price below SMA20; SMA20 above SMA50; RSI bear...
1106,-2,NEUTRAL,Price below SMA20; SMA20 above SMA50; RSI bear...
1107,-2,NEUTRAL,Price below SMA20; SMA20 above SMA50; RSI bear...
1108,-2,NEUTRAL,Price below SMA20; SMA20 above SMA50; RSI bear...


Scanning MSFT...
Q-SCANNER — SIGNAL ENGINE V3


Price,Score,Signal,Reasons
1099,1,NEUTRAL,Price above SMA20; SMA20 above SMA50; RSI over...
1100,1,NEUTRAL,Price above SMA20; SMA20 above SMA50; RSI over...
1101,2,NEUTRAL,Price above SMA20; SMA20 above SMA50; RSI over...
1102,2,NEUTRAL,Price above SMA20; SMA20 above SMA50; RSI over...
1103,2,NEUTRAL,Price above SMA20; SMA20 above SMA50; RSI over...
1104,2,NEUTRAL,Price above SMA20; SMA20 above SMA50; RSI over...
1105,2,NEUTRAL,Price above SMA20; SMA20 above SMA50; RSI over...
1106,2,NEUTRAL,Price above SMA20; SMA20 above SMA50; RSI over...
1107,2,NEUTRAL,Price above SMA20; SMA20 above SMA50; RSI over...
1108,2,NEUTRAL,Price above SMA20; SMA20 above SMA50; RSI over...


Scanning NVDA...
Q-SCANNER — SIGNAL ENGINE V3


...

Price,Score,Signal,Reasons
1099,2,NEUTRAL,Price above SMA20; SMA20 below SMA50; RSI bull...
1100,2,NEUTRAL,Price above SMA20; SMA20 below SMA50; RSI bull...
1101,2,NEUTRAL,Price above SMA20; SMA20 below SMA50; RSI bull...
1102,2,NEUTRAL,Price above SMA20; SMA20 below SMA50; RSI bull...
1103,2,NEUTRAL,Price above SMA20; SMA20 below SMA50; RSI bull...
1104,2,NEUTRAL,Price above SMA20; SMA20 below SMA50; RSI bull...
1105,-2,NEUTRAL,Price below SMA20; SMA20 below SMA50; RSI bear...
1106,-2,NEUTRAL,Price below SMA20; SMA20 below SMA50; RSI bear...
1107,-2,NEUTRAL,Price below SMA20; SMA20 below SMA50; RSI bear...
1108,-4,SELL,Price below SMA20; SMA20 below SMA50; RSI bear...


Scanning TSLA...
Q-SCANNER — SIGNAL ENGINE V3


Price,Score,Signal,Reasons
1099,-4,SELL,Price below SMA20; SMA20 below SMA50; RSI bear...
1100,-4,SELL,Price below SMA20; SMA20 below SMA50; RSI bear...
1101,-4,SELL,Price below SMA20; SMA20 below SMA50; RSI bear...
1102,-4,SELL,Price below SMA20; SMA20 below SMA50; RSI bear...
1103,-2,NEUTRAL,Price below SMA20; SMA20 below SMA50; RSI bear...
1104,-2,NEUTRAL,Price below SMA20; SMA20 below SMA50; RSI bear...
1105,-2,NEUTRAL,Price below SMA20; SMA20 below SMA50; RSI bear...
1106,-2,NEUTRAL,Price below SMA20; SMA20 below SMA50; RSI bear...
1107,0,NEUTRAL,Price above SMA20; SMA20 below SMA50; RSI bear...
1108,0,NEUTRAL,Price above SMA20; SMA20 below SMA50; RSI bear...



Q-SCANNER V4 — CURRENT MARKET SIGNALS


Price,Ticker,Close,SMA_20,SMA_50,RSI,Momentum,Score,Signal,Reasons
0,AMZN,266.429993,266.884500,251.659200,55.852184,1.439177,2,NEUTRAL,Price below SMA20; SMA20 above SMA50; RSI bull...
1,TSLA,348.750000,338.818997,360.473798,50.888322,1.893245,2,NEUTRAL,Price above SMA20; SMA20 below SMA50; RSI bull...
2,META,578.020020,575.769498,592.144194,49.561108,-2.005587,-2,NEUTRAL,Price above SMA20; SMA20 below SMA50; RSI bear...
3,AAPL,319.700012,309.794240,311.811248,57.479998,4.501036,1,NEUTRAL,Price above SMA20; SMA20 below SMA50; RSI bull...
4,MSFT,513.530029,492.748413,429.973376,73.533335,3.854817,1,NEUTRAL,Price above SMA20; SMA20 above SMA50; RSI over...
5,NVDA,217.550003,218.047501,208.419999,52.343541,-3.379819,0,NEUTRAL,Price below SMA20; SMA20 above SMA50; RSI bull...
6,GOOGL,346.589996,350.271498,349.777599,49.261840,0.199480,0,NEUTRAL,Price below SMA20; SMA20 above SMA50; RSI bear...
